# **Silver Layer - Handling/ cleaning data**


### **Imporing libraries**

In [869]:
import pandas as pd
import numpy as np

### **Reading data**

Read bronze layer data

In [870]:
Silver_df = pd.read_csv("/Users/folder/project_real_estate/data/raw/Bronze.csv")

### **Configure pandas display settings**

In [871]:
pd.options.display.float_format = '{:,.2f}'.format
pd.options.display.max_rows = None
pd.options.display.max_columns = None

In [872]:
Silver_df.head()

,url,price,description,location,type,size,bedrooms,bathrooms,available_from,payment_method,down_payment
0,https://www.propertyfinder.eg/en/plp/buy/chale...,"8,000,000",OWN A CHALET IN EL GOUNA WITH A PRIME LOCATION...,"Swan Lake Gouna, Al Gouna, Hurghada, Red Sea",Chalet,732 sqft / 68 sqm,1+ Maid,1,31 Aug 2025,Cash,"1,200,000 EGP"
1,https://www.propertyfinder.eg/en/plp/buy/villa...,"25,000,000","For sale, a villa with immediate delivery in C...","Karmell, New Zayed City, Sheikh Zayed City, Giza",Villa,"2,368 sqft / 220 sqm",4,4,2 Sep 2025,Cash,"2,100,000 EGP"
2,https://www.propertyfinder.eg/en/plp/buy/chale...,"15,135,000","With a down payment of EGP 1,513,000, a fully ...","Azha North, Ras Al Hekma, North Coast",Chalet,"1,270 sqft / 118 sqm",2,2,19 Aug 2025,Cash,"1,513,000 EGP"
3,https://www.propertyfinder.eg/en/plp/buy/apart...,"12,652,000",Own an apartment in New Cairo with a minimal d...,"Taj City, 5th Settlement Compounds, The 5th Se...",Apartment,"1,787 sqft / 166 sqm",3,2,26 Aug 2025,Installments,"1,260,000 EGP"
4,https://www.propertyfinder.eg/en/plp/buy/villa...,"45,250,000",Project: Granville\nLocation: Fifth Settlement...,"Granville, New Capital City, Cairo",Villa,"4,306 sqft / 400 sqm",7,7,2 Sep 2025,Cash,"2,262,500 EGP"


### **Cleaning data**

#### Check duplicate rows

In [873]:
# Total duplicate rows
duplicate_rows = Silver_df.duplicated().sum()

print(f"Duplicate rows: {duplicate_rows}")

Duplicate rows: 0


#### URL

In [874]:
print("Duplicate rows:", Silver_df.duplicated().sum())

print("Duplicate URLs:", Silver_df["url"].duplicated().sum())

Duplicate rows: 0
Duplicate URLs: 0


In [875]:
# Convert to string and remove extra spaces
Silver_df["url"] = (
    Silver_df["url"]
    .astype("string")
    .str.strip()
)

# Ensure URLs start with https://
Silver_df["url"] = Silver_df["url"].str.replace(
    r"^http://",
    "https://",
    regex=True
)

missing_url = Silver_df["url"].isna() | (Silver_df["url"].str.strip() == "")

# Remove duplicate listings based on URL
# Rows with a valid URL
with_url = Silver_df[~missing_url]

# Rows without a URL
without_url = Silver_df[missing_url]

# Remove duplicate URLs only
with_url = with_url.drop_duplicates(subset=["url"])

# Combine everything back
Silver_df = pd.concat([with_url, without_url], ignore_index=True)


#### price

In [876]:
Silver_df["price"] = (
    Silver_df["price"]
    .str.replace(",", "", regex=False)
    .astype("Int64")
)

In [877]:
Silver_df.nsmallest(
    10,
    "price"
)[["price"]]

,price
15671,186900
454,200000
3538,202500
7507,233766
13041,285000
7489,300000
132,325000
190,325000
438,325000
2605,325000


In [878]:
Silver_df.nlargest(
    10,
    "price"
)[["price"]]

,price
14762,840000000
12670,650000000
18560,453000000
12051,400000000
13447,400000000
15066,400000000
9871,375000000
13068,375000000
12370,374000000
11523,360000000


In [879]:
Silver_df["price"].describe(percentiles=[0.01,0.05,0.25,0.5,0.75,0.95,0.99])

count        19,385.00
mean     16,415,306.81
std      23,697,908.14
min         186,900.00
1%        1,370,000.00
5%        2,700,000.00
25%       6,000,000.00
50%      10,345,550.00
75%      18,169,000.00
95%      47,852,500.00
99%     100,000,000.00
max     840,000,000.00
Name: price, dtype: Float64

#### description

1 *Remove extra spaces*
2 *Remove repeated new lines*
3 *Strip whitespace*

In [880]:
Silver_df["description"] = (
    Silver_df["description"]
    .str.replace("\n", " ", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

#### Location

In [881]:
def split_location(location):
    if pd.isna(location):
        return pd.Series(
            [np.nan, np.nan, np.nan, np.nan],
            index=["compound", "district", "city", "governorate"]
        )

    parts = [p.strip() for p in str(location).split(",")]

    # -------------------------------------
    # Pattern: Compound, Governorate
    # Example:
    # Madinaty, Cairo
    # -------------------------------------
    if len(parts) == 2:
        return pd.Series(
            [parts[0], np.nan, np.nan, parts[1]],
            index=["compound", "district", "city", "governorate"]
        )

    # -------------------------------------
    # Pattern: Compound, City, Governorate
    # Example:
    # Marassi, Sidi Abdel Rahman, North Coast
    #
    # OR
    #
    # Compound, District, Governorate
    # Example:
    # Silver Sands, Qesm Marsa Matrouh, North Coast
    # -------------------------------------
    elif len(parts) == 3:

        second = parts[1].lower()

        if second.startswith("qesm"):
            district = parts[1]
            city = np.nan
        else:
            district = np.nan
            city = parts[1]

        return pd.Series(
            [parts[0], district, city, parts[2]],
            index=["compound", "district", "city", "governorate"]
        )

    # -------------------------------------
    # Pattern: Compound, District, City, Governorate
    # Example:
    # Mangroovy Residence, Al Gouna, Hurghada, Red Sea
    # -------------------------------------
    elif len(parts) == 4:

        return pd.Series(
            [parts[0], parts[1], parts[2], parts[3]],
            index=["compound", "district", "city", "governorate"]
        )

    # -------------------------------------
    # Pattern: Compound, District, Area..., Governorate
    # Example:
    # Mountain View iCity,
    # 5th Settlement Compounds,
    # The 5th Settlement,
    # New Cairo City,
    # Cairo
    # -------------------------------------
    elif len(parts) > 4:

        city = ", ".join(parts[2:-1])

        return pd.Series(
            [parts[0], parts[1], city, parts[-1]],
            index=["compound", "district", "city", "governorate"]
        )

    # -------------------------------------
    # Pattern: Compound only
    # -------------------------------------
    else:

        return pd.Series(
            [parts[0], np.nan, np.nan, np.nan],
            index=["compound", "district", "city", "governorate"]
        )


Silver_df[
    ["compound", "district", "city", "governorate"]
] = Silver_df["location"].apply(split_location)

In [882]:
Silver_df = Silver_df.drop(columns=["location"])

In [883]:
Silver_df["compound"].value_counts(dropna=False)

compound
Marassi                                                   433
Mountain View iCity                                       393
Madinaty                                                  368
Hyde Park                                                 359
Palm Hills New Cairo                                      322
Silver Sands                                              299
O West                                                    283
Sarai                                                     267
Mountain View iCity October                               257
Mivida                                                    242
Mountain View                                             239
Mangroovy Residence                                       228
Taj City                                                  225
Azha North                                                221
Mountain View Hyde Park                                   218
Mesca                                                     200

#### type


In [884]:
Silver_df["type"].value_counts(dropna=False)

type
Apartment          8355
Chalet             4038
Villa              3570
Townhouse          1335
Twin House          834
Duplex              622
Penthouse           569
iVilla              268
Hotel Apartment     104
NaN                  77
Land                 63
Cabin                38
Palace               23
Whole Building       14
Roof                  6
Full Floor            4
Bulk Sale Unit        3
Bungalow              1
Name: count, dtype: int64

In [885]:
Silver_df["type"] = (
    Silver_df["type"]
    .astype("string")
    .str.strip()
    .str.title()
    .replace({
        "Ivilla": "iVilla"
    })
    .astype("category")
)

In [886]:
Silver_df["type"].value_counts(dropna=False)

type
Apartment          8355
Chalet             4038
Villa              3570
Townhouse          1335
Twin House          834
Duplex              622
Penthouse           569
iVilla              268
Hotel Apartment     104
<NA>                 77
Land                 63
Cabin                38
Palace               23
Whole Building       14
Roof                  6
Full Floor            4
Bulk Sale Unit        3
Bungalow              1
Name: count, dtype: int64

In [887]:
Silver_df["type"] = (
    Silver_df["type"]
    .cat.add_categories(["Unknown"])
    .fillna("Unknown")
)

#### size

In [888]:
# Extract square feet
Silver_df["size_sqft"] = (
    Silver_df["size"]
    .str.extract(r'([\d,]+)\s*sqft')[0]
    .str.replace(",", "", regex=False)
    .astype("Int64")
)

# Extract square meters
Silver_df["size_sqm"] = (
    Silver_df["size"]
    .str.extract(r'/\s*([\d,]+)\s*sqm')[0]
    .str.replace(",", "", regex=False)
    .astype("Int64")
)

In [889]:
print(Silver_df[["size", "size_sqft", "size_sqm"]].head())

                   size  size_sqft  size_sqm
0     732 sqft / 68 sqm        732        68
1  2,368 sqft / 220 sqm       2368       220
2  1,270 sqft / 118 sqm       1270       118
3  1,787 sqft / 166 sqm       1787       166
4  4,306 sqft / 400 sqm       4306       400


In [890]:
Silver_df["size_sqft"].describe()

count        19,847.00
mean         11,135.95
std       1,208,324.15
min              11.00
25%           1,367.00
50%           1,830.00
75%           2,594.00
max     170,195,636.00
Name: size_sqft, dtype: Float64

In [891]:
Silver_df[Silver_df["size_sqm"] < 20]

,url,price,description,type,size,bedrooms,bathrooms,available_from,payment_method,down_payment,compound,district,city,governorate,size_sqft,size_sqm
2693,https://www.propertyfinder.eg/en/plp/buy/villa...,39000000,"Villa for sale in Madinaty, one of the best ar...",Villa,22 sqft / 2 sqm,5+ Maid,4,6 Aug 2025,Cash,"12,000,000 EGP",Madinaty,NaN,NaN,Cairo,22,2
14334,https://www.propertyfinder.eg/en/plp/buy/chale...,1081500,دلوقتي عندك فرصة تمتلك ستوديو أنيق في جولف مار...,Chalet,11 sqft / 1 sqm,studio,1,28 Aug 2025,Cash,NaN,Al Alamein,NaN,NaN,North Coast,11,1


In [892]:
Silver_df = Silver_df[Silver_df["size_sqm"] >= 20]

In [893]:
print("Missing sqft:", Silver_df["size_sqft"].isna().sum())
print("Missing sqm :", Silver_df["size_sqm"].isna().sum())

Missing sqft: 0
Missing sqm : 0


In [894]:
missing_size = Silver_df["size_sqm"].isna().sum()

print(f"Missing size records: {missing_size}")
print(f"Missing percentage: {missing_size / len(Silver_df) * 100:.2f}%")

Missing size records: 0
Missing percentage: 0.00%


In [895]:
Silver_df.drop(columns=["size"], inplace=True)

In [896]:
print(Silver_df[["size_sqft", "size_sqm"]].dtypes)

size_sqft    Int64
size_sqm     Int64
dtype: object


In [897]:
Silver_df.nsmallest(
    10,
    "size_sqm"
)[["size_sqm", "size_sqm", "compound", "city"]]

,size_sqm,size_sqm,compound,city
14774,20,20,Ivoire,Sheikh Zayed City
9824,27,27,Zahra,Sidi Abdel Rahman
4803,29,29,Valencia Valley,New Cairo City
185,30,30,Golf Porto Marina,Al Alamein
527,30,30,Golf Porto Marina,Al Alamein
1367,30,30,Glee,NaN
2877,30,30,Golf Porto Marina,Al Alamein
2999,30,30,Golf Porto Marina,Al Alamein
3014,30,30,Golf Porto Marina,Al Alamein
4269,30,30,Golf Porto Marina,Al Alamein


In [898]:
Silver_df.nlargest(
    10,
    "size_sqm"
)[["size_sqm", "price", "compound", "city"]]

,size_sqm,price,compound,city
4410,15811692,11692311,Sarai,Mostakbal City - Future City
8277,302400,<NA>,The 6th Settlement,New Cairo City
15510,96106,6550000,Mountain View,Ras Al Hekma
794,27500,300000000,Green Revolution,Sheikh Zayed City
15204,21000,130200000,New Zayed City,Sheikh Zayed City
9669,9500,<NA>,Hood 11 Side St.,6 October City
18512,8400,<NA>,Green Revolution,Sheikh Zayed City
8174,6779,3500000,Sidi Gaber St.,Hay Sharq
13451,4200,<NA>,Green Revolution,Sheikh Zayed City
12670,3800,650000000,Katameya Heights,"El Katameya, New Cairo City"


#### bedrooms

In [899]:
Silver_df["bedrooms"].value_counts(dropna=False)

bedrooms
3               4959
3+ Maid         3737
2               2849
4+ Maid         2126
2+ Maid         1748
4               1126
5+ Maid          785
1                751
1+ Maid          447
5                325
studio           279
6+ Maid          254
7+ Maid          128
6                 93
NaN               67
7++ Maid          67
studio+ Maid      48
7                 33
7+                23
Name: count, dtype: int64

In [900]:
Silver_df = (
    Silver_df
    .rename(columns={"bedrooms": "bedrooms_raw"})
    .assign(
        # Maid room indicator
        maid_room=lambda df: (
            df["bedrooms_raw"]
            .astype("string")
            .str.contains("Maid", case=False, na=False)
            .astype("boolean")
        ),

        # Extract bedroom count
        bedrooms=lambda df: (
            df["bedrooms_raw"]
            .astype("string")
            .str.extract(r"(\d+)")[0]
            .astype("Int64")
        )
    )
)

# Convert studios to 0 bedrooms
Silver_df.loc[
    Silver_df["bedrooms_raw"].str.contains("studio", case=False, na=False),
    "bedrooms"
] = 0

In [901]:
# Drop the original column after validation
Silver_df = Silver_df.drop(columns="bedrooms_raw")

In [902]:
Silver_df["bedrooms"].value_counts(dropna=False)

bedrooms
3       8696
2       4597
4       3252
1       1198
5       1110
6        347
0        327
7        251
<NA>      67
Name: count, dtype: Int64

#### bathrooms

In [903]:
Silver_df["bathrooms"].value_counts(dropna=False)

bathrooms
3       6562
2       5529
4       3280
1       2096
5       1313
6        463
7        223
7+       159
NaN       63
2.0       48
3.0       48
1.0       21
4.0       19
5.0       11
none       5
6.0        3
7.0        2
Name: count, dtype: int64

In [904]:
Silver_df = (
    Silver_df
    .rename(columns={"bathrooms": "bathrooms_raw"})
    .assign(
        bathrooms_plus=lambda df: (
            df["bathrooms_raw"]
            .astype("string")
            .str.contains(r"\+", regex=True, na=False)
            .astype("boolean")
        ),

        bathrooms=lambda df: (
            df["bathrooms_raw"]
            .astype("string")
            .str.strip()
            .replace({"none": pd.NA})
            .str.extract(r"(\d+)")[0]
            .astype("Int64")
        )
    )
)

In [905]:
# Drop the original column after validation
Silver_df = Silver_df.drop(columns="bathrooms_raw")

In [906]:
Silver_df["bathrooms"].value_counts(dropna=False)

bathrooms
3       6610
2       5577
4       3299
1       2117
5       1324
6        466
7        384
<NA>      68
Name: count, dtype: Int64

#### available_from

In [907]:
Silver_df.head()

,url,price,description,type,available_from,payment_method,down_payment,compound,district,city,governorate,size_sqft,size_sqm,maid_room,bedrooms,bathrooms_plus,bathrooms
0,https://www.propertyfinder.eg/en/plp/buy/chale...,8000000,OWN A CHALET IN EL GOUNA WITH A PRIME LOCATION...,Chalet,31 Aug 2025,Cash,"1,200,000 EGP",Swan Lake Gouna,Al Gouna,Hurghada,Red Sea,732,68,True,1,False,1
1,https://www.propertyfinder.eg/en/plp/buy/villa...,25000000,"For sale, a villa with immediate delivery in C...",Villa,2 Sep 2025,Cash,"2,100,000 EGP",Karmell,New Zayed City,Sheikh Zayed City,Giza,2368,220,False,4,False,4
2,https://www.propertyfinder.eg/en/plp/buy/chale...,15135000,"With a down payment of EGP 1,513,000, a fully ...",Chalet,19 Aug 2025,Cash,"1,513,000 EGP",Azha North,NaN,Ras Al Hekma,North Coast,1270,118,False,2,False,2
3,https://www.propertyfinder.eg/en/plp/buy/apart...,12652000,Own an apartment in New Cairo with a minimal d...,Apartment,26 Aug 2025,Installments,"1,260,000 EGP",Taj City,5th Settlement Compounds,"The 5th Settlement, New Cairo City",Cairo,1787,166,False,3,False,2
4,https://www.propertyfinder.eg/en/plp/buy/villa...,45250000,"Project: Granville Location: Fifth Settlement,...",Villa,2 Sep 2025,Cash,"2,262,500 EGP",Granville,NaN,New Capital City,Cairo,4306,400,False,7,False,7


In [908]:
Silver_df["available_from"] = pd.to_datetime(
    Silver_df["available_from"],
    format="%d %b %Y",
    errors="coerce"
)

In [909]:
print(Silver_df["available_from"].dtype)

datetime64[ns]


#### payment_method

In [910]:
Silver_df["payment_method"].value_counts(dropna=False)

payment_method
Cash            15519
Installments     3862
NaN               464
Name: count, dtype: int64

In [911]:
Silver_df["payment_method"] = (
    Silver_df["payment_method"]
    .astype("string")
    .str.strip()
    .str.title()
)

In [912]:
Silver_df["payment_method"] = Silver_df["payment_method"].astype("category")

In [913]:
Silver_df["payment_method"].dtype

CategoricalDtype(categories=['Cash', 'Installments'], ordered=False, categories_dtype=string)

In [914]:
Silver_df["payment_method"] = (
    Silver_df["payment_method"]
    .cat.add_categories(["Unknown"])
    .fillna("Unknown")
)

In [915]:
Silver_df["payment_method"].value_counts(dropna=False)

payment_method
Cash            15519
Installments     3862
Unknown           464
Name: count, dtype: int64

#### down_payment

In [916]:
Silver_df["down_payment"] = (
    Silver_df["down_payment"]
        .astype("string")
        .str.replace("EGP", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
)

Silver_df["down_payment"] = pd.to_numeric(
    Silver_df["down_payment"],
    errors="coerce"
).astype("Int64")

In [917]:
Silver_df["down_payment"].describe()

count         5,444.00
mean      1,979,201.60
std       3,866,547.91
min               1.00
25%         483,541.25
50%         941,500.00
75%       2,032,125.00
max     150,000,000.00
Name: down_payment, dtype: Float64

In [918]:
Silver_df["down_payment"].value_counts(dropna=False)

down_payment
<NA>         14401
100000         203
5              131
1              102
500000          94
1000000         93
1500000         76
800000          67
300000          62
2000000         61
600000          61
750000          58
550000          55
650000          54
900000          54
3000000         54
400000          53
700000          51
1250000         50
1200000         46
450000          39
4000000         39
2500000         37
850000          36
1400000         33
1150000         33
1300000         33
1100000         32
2800000         29
1050000         27
1600000         26
10              26
3500000         25
1800000         23
20              23
2400000         23
950000          23
350000          22
6000000         22
250000          21
1900000         21
9000000         21
1700000         21
5000000         20
10000000        19
1350000         19
200000          18
1450000         18
2100000         18
625000          18
4500000         17
3200000         15

#### cheack before save 

In [919]:
Silver_df.isna().sum().sort_values(ascending=False)

down_payment      14401
district           4978
city               2127
available_from      586
price               462
bathrooms            68
bedrooms             67
compound             14
governorate          14
description           1
payment_method        0
type                  0
size_sqft             0
size_sqm              0
maid_room             0
bathrooms_plus        0
url                   0
dtype: int64

In [920]:
Silver_df[[
    "price",
    "size_sqft",
    "size_sqm",
    "bedrooms",
    "bathrooms",
    "down_payment"
]].describe()

,price,size_sqft,size_sqm,bedrooms,bathrooms,down_payment
count,"19,383.00","19,845.00","19,845.00","19,778.00","19,777.00","5,444.00"
mean,"16,414,932.73","11,137.07","1,034.67",2.98,2.95,"1,979,201.60"
std,"23,698,319.60","1,208,385.04","112,262.64",1.17,1.29,"3,866,547.91"
min,"186,900.00",215.00,20.00,0.00,1.00,1.00
25%,"6,000,000.00","1,367.00",127.00,2.00,2.00,"483,541.25"
50%,"10,345,550.00","1,830.00",170.00,3.00,3.00,"941,500.00"
75%,"18,161,726.50","2,594.00",241.00,4.00,4.00,"2,032,125.00"
max,"840,000,000.00","170,195,636.00","15,811,692.00",7.00,7.00,"150,000,000.00"


In [921]:
Silver_df["bathrooms_plus"].value_counts()

bathrooms_plus
False    19686
True       159
Name: count, dtype: Int64

In [922]:
Silver_df["maid_room"].value_counts()

maid_room
False    10505
True      9340
Name: count, dtype: Int64

In [923]:
Silver_df["payment_method"].value_counts()

payment_method
Cash            15519
Installments     3862
Unknown           464
Name: count, dtype: int64

In [924]:
Silver_df["type"].value_counts()

type
Apartment          8355
Chalet             4037
Villa              3569
Townhouse          1335
Twin House          834
Duplex              622
Penthouse           569
iVilla              268
Hotel Apartment     104
Land                 63
Cabin                38
Palace               23
Whole Building       14
Roof                  6
Full Floor            4
Bulk Sale Unit        3
Bungalow              1
Unknown               0
Name: count, dtype: int64

In [925]:
print(Silver_df["available_from"].min())
print(Silver_df["available_from"].max())

2023-06-14 00:00:00
2027-10-31 00:00:00


In [926]:
category_cols = [
    "type",
    "payment_method",
    "governorate"
]

for col in category_cols:
    Silver_df[col] = Silver_df[col].astype("category")

In [927]:
Silver_df.loc[
    Silver_df["available_from"] == Silver_df["available_from"].max(),
    ["available_from", "type", "price", "compound", "city"]
]

,available_from,type,price,compound,city
414,2027-10-31,Apartment,5000000,Al Thawra El Khadra,6 October City


In [928]:
Silver_df.loc[
    Silver_df["available_from"] >= "2027-01-01",
    ["available_from", "compound", "city", "price"]
].sort_values("available_from")

,available_from,compound,city,price
414,2027-10-31,Al Thawra El Khadra,6 October City,5000000


In [929]:
print("Earliest availability:", Silver_df["available_from"].min())
print("Latest availability:", Silver_df["available_from"].max())

Earliest availability: 2023-06-14 00:00:00
Latest availability: 2027-10-31 00:00:00


In [930]:
future = Silver_df[
    Silver_df["available_from"] > pd.Timestamp.today().normalize()
]

print(f"Future listings: {len(future)}")
future[["available_from", "compound", "city", "price"]].head()

Future listings: 114


,available_from,compound,city,price
50,2026-08-21,Atalantis,Hurghada,3017400
333,2026-08-31,Princess Resort,Hurghada,2240000
375,2026-08-12,Magawish Resort,Hurghada,3042000
380,2026-08-31,Ghazala Bay,NaN,13504000
390,2026-08-29,Mountain View 1.1,"The 5th Settlement, New Cairo City",26100000


In [931]:
print("Availability Date Range")
print("-----------------------")
print("Minimum:", Silver_df["available_from"].min())
print("Maximum:", Silver_df["available_from"].max())

future = Silver_df[
    Silver_df["available_from"] > pd.Timestamp.today().normalize()
]

print(f"\nFuture listings: {len(future)}")
print(f"Percentage: {len(future)/len(Silver_df)*100:.2f}%")

Availability Date Range
-----------------------
Minimum: 2023-06-14 00:00:00
Maximum: 2027-10-31 00:00:00

Future listings: 114
Percentage: 0.57%


In [932]:
Silver_df.head()

,url,price,description,type,available_from,payment_method,down_payment,compound,district,city,governorate,size_sqft,size_sqm,maid_room,bedrooms,bathrooms_plus,bathrooms
0,https://www.propertyfinder.eg/en/plp/buy/chale...,8000000,OWN A CHALET IN EL GOUNA WITH A PRIME LOCATION...,Chalet,2025-08-31,Cash,1200000,Swan Lake Gouna,Al Gouna,Hurghada,Red Sea,732,68,True,1,False,1
1,https://www.propertyfinder.eg/en/plp/buy/villa...,25000000,"For sale, a villa with immediate delivery in C...",Villa,2025-09-02,Cash,2100000,Karmell,New Zayed City,Sheikh Zayed City,Giza,2368,220,False,4,False,4
2,https://www.propertyfinder.eg/en/plp/buy/chale...,15135000,"With a down payment of EGP 1,513,000, a fully ...",Chalet,2025-08-19,Cash,1513000,Azha North,NaN,Ras Al Hekma,North Coast,1270,118,False,2,False,2
3,https://www.propertyfinder.eg/en/plp/buy/apart...,12652000,Own an apartment in New Cairo with a minimal d...,Apartment,2025-08-26,Installments,1260000,Taj City,5th Settlement Compounds,"The 5th Settlement, New Cairo City",Cairo,1787,166,False,3,False,2
4,https://www.propertyfinder.eg/en/plp/buy/villa...,45250000,"Project: Granville Location: Fifth Settlement,...",Villa,2025-09-02,Cash,2262500,Granville,NaN,New Capital City,Cairo,4306,400,False,7,False,7


### **Save Silver Layer**

In [933]:
Silver_df.to_csv(
    "/Users/folder/project_real_estate/data/raw/Silver.csv",
    index=False
)

print("Silver Layer Created Successfully")

Silver Layer Created Successfully
